Выполнил: Рассохин Владислав Романович

Бизнес-контекст

Интернет-магазин одежды и аксессуаров собирает каталог товаров из разных внутренних систем. У каждого товара есть карточка: фотография, текстовое название, категория, цвет, сезон и другие атрибуты. Такие данные нужны для поиска, рекомендаций, автоматической модерации и классификации новых товаров.

Проблема
Команда хочет обучить модель, которая по изображению товара и его названию сможет определять тип товара: например, рубашка, часы, спортивная обувь или сумка. Но перед обучением нужно убедиться, что датасет действительно готов: файлы существуют, изображения открываются, классы сбалансированы, текстовые названия заполнены, а пайплайн загрузки данных возвращает тензоры правильной формы.
В рабочих проектах самая частая ошибка — сразу перейти к модели и заметить проблемы уже в момент обучения. Например, часть путей не существует, изображения имеют неожиданный формат, классы перепутаны, текстовые признаки не подготовлены, а DataLoader падает на первом батче.
В этом проекте вы будете действовать как инженер: сначала проверите данные, затем подготовите признаки, затем соберёте воспроизводимый пайплайн.

Бизнес-задача

Подготовить датасет товаров интернет-магазина к обучению модели классификации. Нужно собрать пайплайн, который читает метаданные, проверяет качество данных, обрабатывает изображения и текстовые названия, создаёт PyTorch Dataset, настраивает DataLoader и выдаёт батчи, готовые для обучения модели.

### Этап 1. Провести аудит и валидацию датасета

#### 1 пункт
Загрузите metadata.csv и class_to_idx.csv. Посмотрите размер таблиц, названия колонок и первые строки. Проверьте, какие значения встречаются в колонках split, class_name и label.

In [ ]:
import torch
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

DATA_DIR = Path("fashion_project_dataset")
METADATA_PATH = DATA_DIR / "metadata.csv"
CLASS_PATH = DATA_DIR / "class_to_idx.csv"


SEED = 42


def set_seed(seed=SEED):
    """Фиксирует случайность для воспроизводимого результата."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()

print(f"Seed установлен: {SEED}")


print(f"PyTorch: {torch.__version__}")
print(f"Датасет найден: {DATA_DIR.exists()}")

Seed установлен: 42
PyTorch: 2.15.0+cpu
Датасет найден: True


In [6]:

df = pd.read_csv(DATA_DIR / "metadata.csv")
class_to_idx = pd.read_csv(DATA_DIR / "class_to_idx.csv")

print("Размер metadata.csv:", df.shape)
print("Столбцы metadata.csv:")
print(df.columns.tolist())

print("\nРазмер class_to_idx.csv:", class_to_idx.shape)
print("Столбцы class_to_idx.csv:")
print(class_to_idx.columns.tolist())

print("\nПервые строки metadata.csv:")
display(df.head())

print("\nТаблица соответствия классов:")
display(class_to_idx)

Размер metadata.csv: (1200, 14)
Столбцы metadata.csv:
['id', 'split', 'class_name', 'label', 'image_path', 'productDisplayName', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage']

Размер class_to_idx.csv: (4, 2)
Столбцы class_to_idx.csv:
['class_name', 'label']

Первые строки metadata.csv:


,id,split,class_name,label,image_path,productDisplayName,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage
0,29257,train,Sports Shoes,2,images/train/Sports Shoes/29257.jpg,ADIDAS Men Duramo 4M Grey Sports Shoes,Men,Footwear,Shoes,Sports Shoes,Grey,Summer,2012.0,Sports
1,21481,train,Watches,3,images/train/Watches/21481.jpg,Ice Men XXL White Watch,Men,Accessories,Watches,Watches,White,Winter,2016.0,Casual
2,34637,train,Sports Shoes,2,images/train/Sports Shoes/34637.jpg,Lotto Men Vienna III White Sports Shoes,Men,Footwear,Shoes,Sports Shoes,White,Summer,2012.0,Sports
3,1787,train,Sports Shoes,2,images/train/Sports Shoes/1787.jpg,Lotto Men Court Logo White-Silver Shoe,Men,Footwear,Shoes,Sports Shoes,White,Fall,2010.0,Sports
4,14712,train,Shirts,1,images/train/Shirts/14712.jpg,Genesis Off-White Formal Shirt,Men,Apparel,Topwear,Shirts,Off White,Winter,2011.0,Formal



Таблица соответствия классов:


,class_name,label
0,Handbags,0
1,Shirts,1
2,Sports Shoes,2
3,Watches,3


In [ ]:
# проверка значений split, class_name и label

for column in ["split", "class_name", "label"]:
    print(f"\nЗначения в столбце {column}:")
    print(sorted(df[column].unique()))


Значения в столбце split:
['test', 'train', 'val']

Значения в столбце class_name:
['Handbags', 'Shirts', 'Sports Shoes', 'Watches']

Значения в столбце label:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


Таблицы успешно загружены: metadata.csv содержит 1200 записей и 14 признаков, а class_to_idx.csv задаёт соответствие четырёх классов числовым меткам. В датасете присутствуют все ожидаемые разбиения - train, val и test, а метки классов принимают значения от 0 до 3.

#### 2 пункт
Затем изучите распределение объектов по разбиениям и классам. Важно убедиться, что в train, val и test представлены все четыре класса и что распределение соответствует описанию датасета.

In [10]:
distribution = (
    df.groupby(["split", "class_name"])
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "val", "test"])
)

print("Число объектов каждого класса:")
display(distribution)

print("Общее число объектов в каждом разбиении:")
print(df["split"].value_counts())

print("Число объектов по классам:")
print(df["class_name"].value_counts())

Число объектов каждого класса:


class_name,Handbags,Shirts,Sports Shoes,Watches
split,,,,
train,210,210,210,210
val,45,45,45,45
test,45,45,45,45


Общее число объектов в каждом разбиении:
split
train    840
val      180
test     180
Name: count, dtype: int64
Число объектов по классам:
class_name
Sports Shoes    300
Watches         300
Shirts          300
Handbags        300
Name: count, dtype: int64


Во всех разбиениях присутствуют четыре класса. Датасет сбалансирован: в train по 210 объектов каждого класса, в val и test по 45, а всего по 300 объектов каждого класса.

#### 3 пункт 
После этого проверьте связку таблицы и файлов на диске. Для каждого значения из image_path нужно убедиться, что такой файл действительно существует внутри папки датасета. Затем проверьте, что изображения открываются без ошибок.

In [11]:
image_paths = df["image_path"].apply(lambda path: DATA_DIR / path)

missing_paths = image_paths[~image_paths.apply(lambda path: path.exists())]

print("Всего путей:", len(image_paths))
print("Несуществующих файлов:", len(missing_paths))

if len(missing_paths) > 0:
    print("Примеры отсутствующих файлов:")
    print(missing_paths.head())

broken_images = []

for path in image_paths:
    try:
        with Image.open(path) as image:
            image.verify()
    except Exception:
        broken_images.append(str(path))

print("Изображений с ошибкой открытия:", len(broken_images))

if len(broken_images) > 0:
    print("Примеры файлов с ошибкой:")
    print(broken_images[:5])

Всего путей: 1200
Несуществующих файлов: 0
Изображений с ошибкой открытия: 0


#### 4 пункт
Также посмотрите базовые свойства изображений: ширину, высоту и количество каналов. Это поможет понять, с какими исходными данными работают преобразования.

In [12]:
image_info = []

for path in image_paths:
    with Image.open(path) as image:
        image_info.append({
            "width": image.width,
            "height": image.height,
            "mode": image.mode,
            "channels": len(image.getbands())
        })

image_info_df = pd.DataFrame(image_info)

print("Статистика ширины и высоты:")
display(image_info_df[["width", "height"]].describe())

print("Размеры изображений:")
print(image_info_df.groupby(["width", "height"]).size())

print("\nЦветовые режимы:")
print(image_info_df["mode"].value_counts())

print("\nКоличество каналов:")
print(image_info_df["channels"].value_counts())

Статистика ширины и высоты:


,width,height
count,1200.000000,1200.000000
mean,59.994167,79.983333
std,0.202073,0.577350
min,53.000000,60.000000
25%,60.000000,80.000000
50%,60.000000,80.000000
75%,60.000000,80.000000
max,60.000000,80.000000


Размеры изображений:
width  height
53     80           1
60     60           1
       80        1198
dtype: int64

Цветовые режимы:
mode
RGB    1184
L        16
Name: count, dtype: int64

Количество каналов:
channels
3    1184
1      16
Name: count, dtype: int64


#### 5 пункт 
Отдельно проверьте текстовое поле productDisplayName: есть ли пропуски, пустые строки, слишком короткие или подозрительные значения.

In [13]:
names = df["productDisplayName"]

missing_names = names.isna().sum()
empty_names = names.fillna("").str.strip().eq("").sum()
short_names = names.fillna("").str.strip().str.len().lt(5).sum()
numeric_names = names.fillna("").str.fullmatch(r"[\d\s]+").sum()

print("Пропуски:", missing_names)
print("Пустые строки:", empty_names)
print("Слишком короткие названия:", short_names)
print("Названия только из цифр:", numeric_names)

print("\nПримеры названий:")
display(df[["productDisplayName", "class_name"]].sample(5, random_state=42))

Пропуски: 0
Пустые строки: 0
Слишком короткие названия: 0
Названия только из цифр: 0

Примеры названий:


,productDisplayName,class_name
1178,John Miller Men Striped Blue Shirt,Shirts
865,ADIDAS Men Runner White & Blue Sports Shoes,Sports Shoes
101,Police Men White Dial Watch PL12963JS-01,Watches
439,Highlander Men Check Red Shirt,Shirts
58,Scullers Men Check Purple Shirts,Shirts


#### 6 пункт

В конце этапа сделайте краткий вывод: пригоден ли датасет для построения пайплайна и какие риски вы нашли. Если проблем не найдено, это тоже результат — подтвердите его проверками.

##### Вывод по этапу 1

Датасет содержит 1200 товаров четырёх классов и корректно разделён на выборки train, val и test. Все классы представлены в каждом разбиении, а распределение объектов сбалансировано, что указано в пункте 1, 2.

Все пути к изображениям существуют, а изображения успешно открываются без ошибок. Большинство изображений имеют размер 60 x 80 и формат RGB, но встречаются 16 одноканальных изображений и 2 изображения нестандартного размера. Поэтому на следующем этапе нужно привести все изображения к RGB и единому размеру.

В поле productDisplayName нет пропусков, пустых, слишком коротких или подозрительных значений. Датасет пригоден для создания пайплайна подготовки изображений и текстовых признаков.

### Этап 2. Подготовить изображения

На втором этапе нужно определить, как изображения будут преобразовываться перед передачей в модель.

#### 1 пункт 

Для обучающей выборки подготовьте преобразования, которые приводят изображение к единому размеру, переводят его в тензор и добавляют несколько случайных аугментаций. Например, можно использовать случайное горизонтальное отражение, небольшой поворот, изменение яркости или контраста.

In [14]:
IMAGE_SIZE = (128, 96)

train_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

print(train_transforms)

Compose(
    Resize(size=(128, 96), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-10.0, 10.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.9, 1.1), contrast=(0.9, 1.1), saturation=None, hue=None)
    ToTensor()
    Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
)


Resize приводит все изображения к одному размеру. Остальные случайные преобразования немного меняют изображения только при обучении, чтобы модель лучше обобщала данные. Для train задан единый размер 128 x 96, перевод изображения в тензор и нормализация. Добавлены умеренные случайные аугментации: отражение, поворот, изменение яркости и контраста.

#### 2 пункт

Для валидационной и тестовой выборок используйте детерминированные преобразования: изменение размера, перевод в тензор и, при необходимости, нормализацию. Случайные аугментации здесь не нужны, потому что val и test должны использоваться для стабильной проверки, а не для искусственного изменения данных.

In [15]:
# Преобразования для валидационной выборки без случайных аугментаций
val_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

# Преобразования для тестовой выборки без случайных аугментаций
test_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

print("Преобразования для val:")
print(val_transforms)

print("\nПреобразования для test:")
print(test_transforms)

Преобразования для val:
Compose(
    Resize(size=(128, 96), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
)

Преобразования для test:
Compose(
    Resize(size=(128, 96), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
)
